# Random Forest for Malware Detection

This notebook implements a Random Forest classifier for binary classification of malware vs goodware.

Random Forest is an ensemble method that combines multiple decision trees to improve prediction accuracy and reduce overfitting.

## Steps:
1. Load and explore the dataset
2. Preprocess the data
3. Split into train/test sets (80/20)
4. Train Random Forest model
5. Evaluate using 10-fold cross-validation
6. Test on hold-out test set
7. Analyze feature importance
8. Save the model

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../dataset/brazilian-malware-dataset/brazilian-malware.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check class distribution
print("Class distribution:")
print(df['Label'].value_counts())
print(f"\nClass balance: {df['Label'].value_counts(normalize=True)}")

## 2. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Label', axis=1)

# Keep only numeric columns
numeric_cols = X.select_dtypes(include=[np.number]).columns
X = X[numeric_cols]

y = df['Label']

# Handle missing values
if X.isnull().sum().sum() > 0:
    print("Filling missing values with 0...")
    X = X.fillna(0)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 3. Train/Test Split (80/20)

In [ ]:
# Split into train and test sets (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

## 4. Feature Scaling

Note: Random Forests don't require scaling, but we'll do it for consistency.

In [ ]:
# Initialize and fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")

## 5. Train Random Forest Model

In [ ]:
# Initialize Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=15,          # Maximum depth of each tree
    min_samples_split=10,  # Minimum samples to split a node
    min_samples_leaf=5,    # Minimum samples in a leaf
    random_state=RANDOM_STATE,
    n_jobs=-1              # Use all CPU cores
)

# Train the model
print("Training Random Forest model...")
rf.fit(X_train_scaled, y_train)
print("Model training complete!")
print(f"Number of trees: {rf.n_estimators}")
print(f"Number of features: {rf.n_features_in_}")

## 6. Cross-Validation Evaluation (10-Fold)

In [ ]:
# Perform 10-fold stratified cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

print("Performing 10-fold cross-validation...")
print("This may take a few minutes...")
cv_results = cross_validate(
    rf, 
    X_train_scaled, 
    y_train,
    cv=cv,
    scoring=['roc_auc', 'accuracy'],
    n_jobs=-1,
    return_train_score=False
)

# Display results
print("\n=== Cross-Validation Results ===")
print(f"AUC: {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")
print(f"Accuracy: {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")

## 7. Final Evaluation on Hold-Out Test Set

In [ ]:
# Make predictions on test set
y_pred = rf.predict(X_test_scaled)
y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_pred_proba)

print("\n=== Test Set Results ===")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"AUC: {test_auc:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Goodware', 'Malware'],
            yticklabels=['Goodware', 'Malware'])
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Goodware', 'Malware']))

## 8. Feature Importance Analysis

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize top features
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances - Random Forest')
plt.tight_layout()
plt.show()

## 9. Out-of-Bag (OOB) Score

Random Forest can use out-of-bag samples for validation without needing a separate validation set.

In [ ]:
# Train a new model with OOB score enabled
rf_oob = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_oob.fit(X_train_scaled, y_train)
print(f"Out-of-Bag Score: {rf_oob.oob_score_:.4f}")
print("\nOOB score provides an unbiased estimate of the model's performance.")

## 10. Save Model

In [ ]:
# Create models directory if it doesn't exist
import os
os.makedirs('../models', exist_ok=True)

# Save the model
with open('../models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf, f)
print("Model saved to ../models/random_forest.pkl")

## Summary

This notebook demonstrated:
1. ✅ Loading and exploring the malware dataset
2. ✅ Preprocessing with proper train/test split
3. ✅ Training a Random Forest classifier
4. ✅ 10-fold stratified cross-validation
5. ✅ Evaluation on hold-out test set
6. ✅ Feature importance analysis
7. ✅ Out-of-Bag score calculation
8. ✅ Model serialization for production use

**Key Results:**
- Cross-Validation AUC: [See results above]
- Test Set AUC: [See results above]
- Test Set Accuracy: [See results above]
- OOB Score: [See results above]

**Advantages of Random Forest:**
- Reduces overfitting compared to single decision trees
- Provides feature importance rankings
- Handles non-linear relationships well
- Robust to outliers and noise
- Can estimate generalization error using OOB samples

**Disadvantages:**
- Can be slower to train and predict than single trees
- Less interpretable than a single decision tree
- Larger model size (multiple trees)